In [1]:
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
import xgboost as xgb
from xgboost import XGBRegressor

In [2]:
import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [3]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/ModelXG/ModelMk1.json")

In [4]:
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = loaded_model.predict(np.array([[s1],[s2],[b1]]).T)[0]
    return np.float64(y_pred)

In [5]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(my_parameters["s1"],my_parameters["s2"],my_parameters["b1"])})

    for _ in range(7): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=3)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
15.791642189025879

Trial 1 =========================================
14.338056564331055

Trial 2 =========================================
15.901840209960938

Trial 3 =========================================
15.125311851501465

Trial 4 =========================================
15.450379371643066

Trial 5 =========================================
15.939385414123535

Trial 6 =========================================
14.624845504760742

Trial 7 =========================================
14.321561813354492

Trial 8 =========================================
14.388423919677734

Trial 9 =========================================
15.352350234985352

Trial 10 =========================================
15.287394523620605

Trial 11 =========================================
14.402359962463379

Trial 12 =========================================
14.808981895446777

Trial 13 =========================================
14.613863945007324

Trial 14 =======

/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 73 =========================================
15.939385414123535

Trial 74 =========================================
15.614129066467285

Trial 75 =========================================
15.614129066467285

Trial 76 =========================================
15.137866973876953

Trial 77 =========================================
15.469331741333008



/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 78 =========================================
14.584916114807129

Trial 79 =========================================
16.60592269897461

Trial 80 =========================================
16.462608337402344

Trial 81 =========================================
14.483154296875

Trial 82 =========================================
15.206700325012207

Trial 83 =========================================
14.495916366577148

Trial 84 =========================================
15.939385414123535

Trial 85 =========================================
15.939385414123535

Trial 86 =========================================
14.589055061340332

Trial 87 =========================================
16.365570068359375

Trial 88 =========================================
14.808981895446777

Trial 89 =========================================
15.939385414123535

Trial 90 =========================================
15.221745491027832

Trial 91 =========================================
17.182085037231445

Trial 92 =

In [6]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 17.497568130493164
Avg = 15.433455123901368
Std = 0.771966504368207


In [7]:
print(y_max_arr.tolist())

[15.791642189025879, 14.338056564331055, 15.901840209960938, 15.125311851501465, 15.450379371643066, 15.939385414123535, 14.624845504760742, 14.321561813354492, 14.388423919677734, 15.352350234985352, 15.287394523620605, 14.402359962463379, 14.808981895446777, 14.613863945007324, 16.236892700195312, 15.610447883605957, 15.968119621276855, 15.614129066467285, 15.757410049438477, 15.103797912597656, 15.680425643920898, 16.934253692626953, 15.258705139160156, 16.236892700195312, 15.610447883605957, 15.721282005310059, 15.497264862060547, 16.08495330810547, 15.939385414123535, 14.548840522766113, 15.14081859588623, 15.939385414123535, 15.6951904296875, 14.18364429473877, 15.469331741333008, 16.418975830078125, 15.614129066467285, 14.617334365844727, 14.337114334106445, 14.24166488647461, 14.808981895446777, 15.90012264251709, 15.564413070678711, 16.623031616210938, 14.434700965881348, 15.939385414123535, 17.496427536010742, 15.939385414123535, 15.454249382019043, 15.701569557189941, 14.728

In [8]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestsOfModelXGB/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestsOfModelXGB/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    13.791407
1    14.298266
2    15.968120
3    16.084953
4    14.096627
..         ...
495  15.103798
496  13.457213
497  15.261514
498  15.995727
499  15.003749

[500 rows x 1 columns]
